# Production Method Comparison (Economic Analysis)

Live economic comparison of vanilla and modded production methods, grouped by slots.

In [18]:
import pandas as pd
import numpy as np

from core.parser.path_resolver import PathResolver
from core.data.building_data import BuildingData
from core.data.goods_data import GoodsData
from core.data.defines_data import DefinesData
from core.data.pop_data import PopData
from analysis.building_levels.building_analysis import load_config

config = load_config()
path_resolver = PathResolver(config['game_path'], config['mod_path'])
goods_data = GoodsData(path_resolver)
building_data = BuildingData(path_resolver)
defines_data = DefinesData(path_resolver)
pop_data = PopData(path_resolver)

goods_data.load_all()
building_data.load_all()
defines_data.load_all()
pop_data.load_all()

pd.options.display.max_columns = None
pd.options.display.precision = 2
pd.options.display.float_format = '{:.2f}'.format

In [19]:
# Resolved food good (good with food > 0) and its price; fallback to defines
food_good_vanilla = goods_data.get_food_good(False)
food_good_modded = goods_data.get_food_good(True)
food_price_vanilla = goods_data.get_food_good_price(False)
food_price_modded = goods_data.get_food_good_price(True)
if food_price_vanilla is None:
    food_price_vanilla = defines_data.get_vanilla_define("NMarket", "FOOD_PRICE")
if food_price_modded is None:
    food_price_modded = defines_data.get_define("NMarket", "FOOD_PRICE")
print(f"Vanilla food good: {food_good_vanilla}, price: {food_price_vanilla}")
print(f"Modded food good:  {food_good_modded}, price: {food_price_modded}")

Vanilla food good: rice, price: 1.0
Modded food good:  victuals, price: 3.0


In [20]:
pop_comparison = pd.merge(
    pop_data.vanilla_df[['pop_food_consumption']].rename(columns={'pop_food_consumption': 'Vanilla_Food'}),
    pop_data.modded_df[['pop_food_consumption']].rename(columns={'pop_food_consumption': 'Modded_Food'}),
    left_index=True,
    right_index=True
)
pop_comparison['Change'] = (pop_comparison['Modded_Food'] / pop_comparison['Vanilla_Food']) - 1
pop_comparison.style.format({'Change': '{:+.1%}'}).background_gradient(subset=['Change'], cmap='RdYlGn_r')

,Vanilla_Food,Modded_Food,Change
name,,,
nobles,20.000000,25.000000,+25.0%
clergy,5.000000,7.000000,+40.0%
burghers,4.000000,7.500000,+87.5%
laborers,1.000000,1.700000,+70.0%
soldiers,5.000000,5.000000,+0.0%
peasants,1.000000,1.000000,+0.0%
tribesmen,0.000000,0.000000,+nan%
slaves,1.000000,0.500000,-50.0%


In [21]:
# goods_data.vanilla_df.columns
# print(goods_data.vanilla_df.shape)
goods_data.vanilla_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).head(15)
# goods_data.vanilla_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).head(40)


# for element in goods_data.vanilla_df.reset_index().name.tolist():
#     print(element)

,method,default_market_price,transport_cost,food
name,,,,
rice,farming,1.00,1.00,10.00
maize,farming,1.00,1.00,8.00
wheat,farming,1.00,1.00,8.00
livestock,farming,1.50,1.00,8.00
potato,farming,1.00,1.00,8.00
fish,gathering,1.00,1.00,5.00
legumes,farming,1.00,2.00,5.00
wool,farming,1.25,1.00,5.00
millet,farming,1.00,1.00,5.00


In [22]:
# goods_data.modded_df.columns
# print(goods_data.modded_df.shape)
# print(goods_data.modded_df.columns)
goods_data.modded_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="food", ascending=False).tail(15)
# goods_data.modded_df[["method", "default_market_price", "transport_cost", "food"]].sort_values(by="transport_cost", ascending=True).head(40)

,method,default_market_price,transport_cost,food
name,,,,
wool,farming,1.25,1.00,0.00
fur,hunting,2.00,1.00,0.00
wild_game,hunting,1.00,1.00,0.00
wheat,farming,1.00,1.00,0.00
maize,farming,1.00,1.00,0.00
rice,farming,1.00,1.00,0.00
fish,gathering,1.00,1.00,0.00
millet,farming,1.00,1.00,0.00
legumes,farming,1.00,2.00,0.00


In [23]:
def get_slot_df(building_name, slot_index, is_modded=True):
    comp = building_data.compare_production_methods(building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data)
    slots = comp['modded_slots'] if is_modded else comp['vanilla_slots']
    
    if slot_index >= len(slots):
        return pd.DataFrame()
    
    # Get building definition for employment info
    b_def = building_data.get_building(building_name) if is_modded else building_data.get_vanilla_building(building_name)
    b_def = b_def or {}
    
    slot = slots[slot_index]
    rows = []
    # System columns that shouldn't be prefixed with In_
    system_cols = [
        'profit', 'profit_margin', 'input_cost', 'output_value', 
        'worker_food_cost', 'is_modifier_output', 'output_price', 'modifier_food_output'
    ]
    
    for pm_name, pm in slot.items():
        row = {
            "Version": "Modded" if is_modded else "Vanilla",
            "PM": pm_name,
            "Pop_Type": b_def.get('pop_type', 'unknown'),
            "Employment (1k)": b_def.get('employment_size_val', 0),
            "EPE": pm.get('epe', 0),
            "Profit": pm.get('profit', 0),
            "Margin": pm.get('profit_margin', 0),
            "Input_Cost": pm.get('input_cost', 0),
            "Output_Val": pm.get('output_value', 0),
            "Worker_Food_Cost": pm.get('worker_food_cost', 0)
        }
        for key, val in pm.items():
            if key in ['produced', 'output', 'category']:
                continue
            
            if key in system_cols:
                row[key] = val
            elif not key.startswith(('profit', 'input_cost', 'output_value', 'worker_food_cost', 'output_price', 'is_modifier_output', 'epe')):
                # Prefix goods with In_
                row[f"In_{key}"] = val
                
        # Handle Output columns
        prod = pm.get('produced')
        if prod and not pd.isna(prod):
            row[f"Out_{prod}"] = pm.get('output', 0)
        modifier_food = pm.get('modifier_food_output', 0)
        if modifier_food and modifier_food != 0:
            row["Out_food"] = modifier_food
            
        rows.append(row)
    return pd.DataFrame(rows)

def analyze_building(building_name):
    comp = building_data.compare_production_methods(building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data)
    num_slots = max(len(comp['vanilla_slots']), len(comp['modded_slots']))
    
    slot_dfs = []
    for i in range(num_slots):
        v_df = get_slot_df(building_name, i, is_modded=False)
        m_df = get_slot_df(building_name, i, is_modded=True)
        combined = pd.concat([v_df, m_df], ignore_index=True)
        if combined.empty: continue
        
        # Cleanup zero columns
        for col in combined.columns:
            if col.startswith(('In_', 'Out_')):
                if (combined[col].fillna(0).abs() < 1e-6).all():
                    combined = combined.drop(columns=[col])
        
        meta = ["Version", "PM", "Pop_Type", "Employment (1k)", "EPE"]
        econ = ["Profit", "Margin", "Input_Cost", "Output_Val", "Worker_Food_Cost", "output_price"]
        goods = sorted([c for c in combined.columns if c not in meta + econ])
        
        final_df = combined[meta + econ + goods].fillna(0)
        
        # Explicitly format all numeric columns to 3 decimal places
        float_cols = final_df.select_dtypes(include=[np.number]).columns.tolist()
        format_dict = {col: "{:.3f}" for col in float_cols}
        if 'Margin' in format_dict: 
            format_dict['Margin'] = "{:+.3%}"
        if 'EPE' in format_dict:
            format_dict['EPE'] = "{:+.3%}"
            
        styled = final_df.style.format(format_dict)
        
        # Add gradients for Margin, Profit and EPE
        if 'Margin' in final_df.columns:
            styled = styled.background_gradient(subset=['Margin'], cmap='RdYlGn', vmin=-0.5, vmax=0.5)
        if 'EPE' in final_df.columns:
            # For EPE, positive means we need more efficiency (bad current state), negative means we are already above break-even
            # So we invert the colormap: negative EPE is Green, positive is Red
            styled = styled.background_gradient(subset=['EPE'], cmap='RdYlGn_r', vmin=-0.5, vmax=0.5)
        if 'Profit' in final_df.columns:
            # For profit, we use a divergent map centered around 0
            # We'll use the max absolute profit to center the scale
            max_prof = final_df['Profit'].abs().max()
            styled = styled.background_gradient(subset=['Profit'], cmap='RdYlGn', vmin=-max_prof, vmax=max_prof)
        
        slot_dfs.append(styled)
    return slot_dfs

## Farming Village

In [24]:
farming_village_analysis = analyze_building("farming_village")
for df in farming_village_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_clay,In_debug_max_profit,In_horses,In_tools,Out_legumes,Out_livestock,Out_maize,Out_millet,Out_olives,Out_potato,Out_rice,Out_wheat,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,farming_village_maintenance,peasants,1.000,-16.667%,0.125,+20.000%,0.625,0.750,0.050,1.500,1.000,0.600,0.000,0.025,0.000,0.500,0.000,0.000,0.000,0.000,0.000,0.000,0.625,0.000,0.750,0.125,0.200,0.050
1,Modded,pp_farming_village_maintenance,peasants,5.000,+63.333%,-0.475,-38.776%,1.225,0.750,0.750,1.500,0.800,0.600,0.000,0.025,0.000,0.500,0.000,0.000,0.000,0.000,0.000,0.000,1.225,0.000,0.750,-0.475,-0.388,0.750
2,Modded,pp_millet_farm_maintenance,peasants,5.000,+96.154%,-0.625,-49.020%,1.275,0.650,0.750,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.650,0.000,0.000,0.000,0.000,1.275,0.000,0.650,-0.625,-0.490,0.750
3,Modded,pp_wheat_farm_maintenance,peasants,5.000,+131.818%,-0.725,-56.863%,1.275,0.550,0.750,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.550,1.275,0.000,0.550,-0.725,-0.569,0.750
4,Modded,pp_maize_farm_maintenance,peasants,5.000,+131.818%,-0.725,-56.863%,1.275,0.550,0.750,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.550,0.000,0.000,0.000,0.000,0.000,1.275,0.000,0.550,-0.725,-0.569,0.750
5,Modded,pp_rice_farm_maintenance,peasants,5.000,+112.500%,-0.675,-52.941%,1.275,0.600,0.750,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.000,0.000,0.000,0.600,0.000,1.275,0.000,0.600,-0.675,-0.529,0.750
6,Modded,pp_legumes_farm_maintenance,peasants,5.000,+112.500%,-0.675,-52.941%,1.275,0.600,0.750,1.000,0.000,0.600,0.075,0.100,0.600,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.275,0.000,0.600,-0.675,-0.529,0.750
7,Modded,pp_potato_farm_maintenance,peasants,5.000,+112.500%,-0.675,-52.941%,1.275,0.600,0.750,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.000,0.000,0.600,0.000,0.000,1.275,0.000,0.600,-0.675,-0.529,0.750
8,Modded,pp_olives_farm_maintenance,peasants,5.000,+131.818%,-0.725,-56.863%,1.275,0.550,0.750,1.000,0.000,0.600,0.075,0.100,0.000,0.000,0.000,0.000,0.550,0.000,0.000,0.000,1.275,0.000,0.550,-0.725,-0.569,0.750


## Fishing Village

In [25]:
fishing_village_analysis = analyze_building("fishing_village")
for df in fishing_village_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_debug_max_profit,In_ivory,In_salt,In_tools,Out_fish,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,fishing_village_maintenance,peasants,1.000,-36.000%,0.180,+56.250%,0.320,0.500,0.050,1.000,1.000,0.000,0.060,0.010,0.500,0.320,0.000,0.500,0.180,0.562,0.050
1,Vanilla,arctic_fishing_village_maintenance,peasants,1.000,-38.000%,0.190,+61.290%,0.310,0.500,0.050,1.000,1.000,0.020,0.045,0.000,0.500,0.310,0.000,0.500,0.190,0.613,0.050
2,Modded,pp_fishing_village_maintenance,peasants,5.000,+176.667%,-0.795,-63.855%,1.245,0.450,0.750,1.000,1.000,0.000,0.075,0.040,0.450,1.245,0.000,0.450,-0.795,-0.639,0.750
3,Modded,pp_arctic_fishing_village_maintenance,peasants,5.000,+166.667%,-0.750,-62.500%,1.200,0.450,0.750,1.000,1.000,0.075,0.030,0.000,0.450,1.200,0.000,0.450,-0.750,-0.625,0.750


## Forest Village

In [26]:
forest_village_analysis = analyze_building("forest_village")
for df in forest_village_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_debug_max_profit,In_sand,In_tar,In_tools,In_weaponry,In_wild_game,Out_food,Out_fur,Out_leather,Out_wild_game,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,forest_tannery,peasants,1.000,-16.923%,0.110,+20.370%,0.540,0.650,0.050,3.000,0.000,0.100,0.020,0.000,0.000,0.400,1.000,0.000,0.200,0.000,0.540,1.000,0.650,0.110,0.204,0.050
1,Vanilla,hunting_lodges,peasants,1.000,-20.000%,0.050,+25.000%,0.200,0.250,0.050,1.000,0.400,0.000,0.000,0.000,0.050,0.000,1.000,0.000,0.000,0.200,0.200,1.000,0.250,0.050,0.250,0.050
2,Modded,pp_forest_tannery,peasants,5.000,+106.667%,-0.640,-51.613%,1.240,0.600,0.750,3.000,0.000,0.100,0.020,0.000,0.000,0.400,0.000,0.000,0.200,0.000,1.240,0.000,0.600,-0.640,-0.516,0.750
3,Modded,pp_hunting_lodges,peasants,5.000,+162.500%,-0.650,-61.905%,1.050,0.400,0.750,1.000,0.400,0.000,0.000,0.000,0.100,0.000,0.000,0.000,0.000,0.400,1.050,0.000,0.400,-0.650,-0.619,0.750
4,Modded,pp_fur_trapping,peasants,5.000,+128.000%,-0.640,-56.140%,1.140,0.500,0.750,2.000,0.400,0.000,0.000,0.050,0.080,0.000,0.000,0.250,0.000,0.000,1.140,0.000,0.500,-0.640,-0.561,0.750


## Fruit Orchard

In [27]:
fruit_orchard_analysis = analyze_building("fruit_orchard")
for df in fruit_orchard_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_lumber,Out_fruit,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,fruit_orchard_maintenance,laborers,1.000,+1.667%,-0.005,-1.639%,0.305,0.300,0.050,1.000,0.170,0.300,0.305,0.000,0.300,-0.005,-0.016,0.050
1,Modded,pp_fruit_orchard_maintenance,peasants,5.000,+104.545%,-0.575,-51.111%,1.125,0.550,0.750,1.000,0.250,0.550,1.125,0.000,0.550,-0.575,-0.511,0.750


## Sheep Farm

In [28]:
sheep_farms_analysis = analyze_building("sheep_farms")
for df in sheep_farms_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_lumber,In_tools,Out_wool,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Vanilla,sheep_farm_maintenance,laborers,1.000,-5.600%,0.035,+5.932%,0.590,0.625,0.050,1.250,0.200,0.080,0.500,0.590,0.000,0.625,0.035,0.059,0.050
1,Modded,pp_sheep_farm_maintenance,peasants,5.000,+78.000%,-0.585,-43.820%,1.335,0.750,0.750,1.250,0.250,0.070,0.600,1.335,0.000,0.750,-0.585,-0.438,0.750


## Cookery

In [29]:
cookery_analysis = analyze_building("cookery")
for df in cookery_analysis: display(df)

,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_fish,In_fruit,In_legumes,In_livestock,In_maize,In_millet,In_pepper,In_potato,In_rice,In_salt,In_wheat,In_wild_game,In_wool,Out_victuals,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Modded,pp_cookery_rice_legumes,peasants,2.000,+44.646%,-2.210,-30.866%,7.160,4.950,0.100,3.000,0.000,0.000,2.810,0.000,0.000,0.000,0.190,0.000,3.300,0.000,0.000,0.000,0.000,1.650,7.160,0.000,4.950,-2.210,-0.309,0.100
1,Modded,pp_cookery_rice_fish,peasants,2.000,+44.646%,-2.210,-30.866%,7.160,4.950,0.100,3.000,2.810,0.000,0.000,0.000,0.000,0.000,0.190,0.000,3.300,0.000,0.000,0.000,0.000,1.650,7.160,0.000,4.950,-2.210,-0.309,0.100
2,Modded,pp_cookery_wheat_livestock,peasants,2.000,+53.535%,-2.650,-34.868%,7.600,4.950,0.100,3.000,0.000,0.000,0.000,2.620,0.000,0.000,0.000,0.000,0.000,0.190,2.620,0.000,0.000,1.650,7.600,0.000,4.950,-2.650,-0.349,0.100
3,Modded,pp_cookery_millet_fish,peasants,2.000,+53.535%,-2.650,-34.868%,7.600,4.950,0.100,3.000,3.740,0.000,0.000,0.000,0.000,2.810,0.000,0.000,0.000,0.190,0.000,0.000,0.000,1.650,7.600,0.000,4.950,-2.650,-0.349,0.100
4,Modded,pp_cookery_maize_livestock,peasants,2.000,+47.778%,-2.365,-32.331%,7.315,4.950,0.100,3.000,0.000,0.000,0.000,2.810,2.060,0.000,0.000,0.940,0.000,0.000,0.000,0.000,0.000,1.650,7.315,0.000,4.950,-2.365,-0.323,0.100
5,Modded,pp_cookery_pemmican,peasants,2.000,+41.818%,-2.070,-29.487%,7.020,4.950,0.100,3.000,0.000,0.370,0.000,3.740,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.940,0.000,1.650,7.020,0.000,4.950,-2.070,-0.295,0.100
6,Modded,pp_cookery_mutton_pease,peasants,2.000,+44.798%,-2.218,-30.938%,7.167,4.950,0.100,3.000,0.000,0.000,1.680,0.000,0.000,0.000,0.000,0.000,0.000,0.190,0.000,0.000,3.550,1.650,7.167,0.000,4.950,-2.218,-0.309,0.100
7,Modded,pp_cookery_labskaus,peasants,2.000,+41.919%,-2.075,-29.537%,7.025,4.950,0.100,3.000,1.500,0.000,0.000,2.990,0.000,0.000,0.000,0.940,0.000,0.000,0.000,0.000,0.000,1.650,7.025,0.000,4.950,-2.075,-0.295,0.100
8,Modded,pp_cookery_surstromming,peasants,2.000,+53.535%,-2.650,-34.868%,7.600,4.950,0.100,3.000,4.680,0.000,0.000,0.000,0.000,0.470,0.000,0.000,0.000,0.470,0.000,0.000,0.000,1.650,7.600,0.000,4.950,-2.650,-0.349,0.100


,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_beer,In_beeswax,In_cloves,In_fruit,In_liquor,In_livestock,In_olives,In_pepper,In_pottery,In_salt,In_sugar,In_tea,In_wine,Out_victuals,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Modded,pp_cookery_no_enhancement,peasants,2.000,+0.000%,-0.100,-100.000%,0.100,0.000,0.100,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.100,0.000,0.000,-0.100,-1.000,0.100
1,Modded,pp_cookery_preserved_rations,peasants,2.000,+63.030%,-2.080,-38.662%,5.380,3.300,0.100,3.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.530,0.550,0.000,0.000,0.000,1.100,5.380,0.000,3.300,-2.080,-0.387,0.100
2,Modded,pp_cookery_sweet_preserves,peasants,2.000,+39.697%,-1.310,-28.416%,4.610,3.300,0.100,3.000,0.000,0.000,0.000,1.540,0.000,0.000,0.000,0.000,0.000,0.000,0.990,0.000,0.000,1.100,4.610,0.000,3.300,-1.310,-0.284,0.100
3,Modded,pp_cookery_spice_blend,peasants,2.000,+69.697%,-2.300,-41.071%,5.600,3.300,0.100,3.000,0.000,0.000,0.550,0.000,0.000,0.000,0.000,0.550,0.000,0.000,0.000,0.000,0.000,1.100,5.600,0.000,3.300,-2.300,-0.411,0.100
4,Modded,pp_cookery_spirited_rations,peasants,2.000,+51.364%,-1.695,-33.934%,4.995,3.300,0.100,3.000,1.210,0.000,0.000,0.000,0.990,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.100,4.995,0.000,3.300,-1.695,-0.339,0.100
5,Modded,pp_cookery_mediterranean_preserves,peasants,2.000,+43.030%,-1.420,-30.085%,4.720,3.300,0.100,3.000,0.000,0.000,0.000,0.000,0.000,0.000,1.320,0.000,0.000,0.000,0.000,0.000,1.650,1.100,4.720,0.000,3.300,-1.420,-0.301,0.100
6,Modded,pp_cookery_honey_confections,peasants,2.000,+46.667%,-1.540,-31.818%,4.840,3.300,0.100,3.000,0.000,1.930,0.000,0.880,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.100,4.840,0.000,3.300,-1.540,-0.318,0.100
7,Modded,pp_cookery_tea_milk_service,peasants,2.000,+43.030%,-1.420,-30.085%,4.720,3.300,0.100,3.000,0.000,0.000,0.000,0.000,0.000,0.880,0.000,0.000,0.000,0.000,0.550,0.550,0.000,1.100,4.720,0.000,3.300,-1.420,-0.301,0.100


,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_steel,Out_victuals,input_cost,modifier_food_output,output_value,profit,profit_margin,worker_food_cost
0,Modded,pp_cookery_basic_packaging,peasants,2.000,+0.000%,-0.100,-100.000%,0.100,0.000,0.100,0.000,0.000,0.000,0.100,0.000,0.000,-0.100,-1.000,0.100
1,Modded,pp_cookery_tin_cans,peasants,2.000,+70.000%,-2.100,-41.176%,5.100,3.000,0.100,3.000,1.000,1.000,5.100,0.000,3.000,-2.100,-0.412,0.100


### Cookery — input goods × slot frequency

For each **slot** (column), each cell counts **how many production methods** in that slot use the **good as a non-zero input** (same identification as the `In_*` columns above). Use this to spot goods that appear in many PMs in one slot (potential market pressure) vs goods that are rare.

The **total** column sums across slots: for a given good, that is the number of PM–good pairs (each PM lives in one slot, so it is the total number of cookery PMs that consume that good).

In [30]:
def _good_info_for_pm_input_key(key: str, is_modded: bool):
    """Match `BuildingData.compare_production_methods` / input cost: only real goods count."""
    if key in ("produced", "output", "category"):
        return None
    g = goods_data.get_good(key) if is_modded else goods_data.get_vanilla_good(key)
    if g is None:
        g = goods_data.get_good(key)
    return g


def goods_slot_input_frequency_matrix(building_name: str, is_modded: bool = True) -> pd.DataFrame:
    """
    Rows: goods. Columns: slot_0, slot_1, ... plus 'total'.
    Values: number of PMs in that slot with a non-zero input amount of that good.
    """
    comp = building_data.compare_production_methods(
        building_name, goods_data=goods_data, defines_data=defines_data, pop_data=pop_data
    )
    slots = comp["modded_slots" if is_modded else "vanilla_slots"]
    pairs = []
    for slot_idx, slot in enumerate(slots):
        for _pm_name, pm in slot.items():
            for key, val in pm.items():
                if key in ("produced", "output", "category"):
                    continue
                if not isinstance(val, (int, float)) or pd.isna(val) or abs(val) < 1e-9:
                    continue
                if _good_info_for_pm_input_key(key, is_modded) is None:
                    continue
                pairs.append((key, slot_idx))
    if not pairs:
        return pd.DataFrame()
    long_df = pd.DataFrame(pairs, columns=["good", "slot"])
    mat = long_df.groupby(["good", "slot"]).size().unstack(fill_value=0).astype(int)
    mat = mat.sort_index()
    mat.columns = [f"slot_{c}" for c in mat.columns]
    mat["total"] = mat.sum(axis=1)
    return mat.sort_values("total", ascending=False)


cookery_input_matrix = goods_slot_input_frequency_matrix("cookery", is_modded=True)
cookery_input_matrix.style.format("{:d}").background_gradient(cmap="Blues", axis=None)

,slot_0,slot_1,slot_2,total
good,,,,
livestock,4,1,0,5
salt,4,1,0,5
fish,4,0,0,4
fruit,1,2,0,3
pepper,2,1,0,3
potato,2,0,0,2
legumes,2,0,0,2
rice,2,0,0,2
millet,2,0,0,2


In [31]:
# Food contribution by input cost: victuals -> 60 food (tavern)
# Attribute % of finished food to each input good by its share of input cost
VICTUALS_TO_FOOD = 60

comp = building_data.compare_production_methods(
    "cookery", goods_data=goods_data, defines_data=defines_data, pop_data=pop_data
)
recipe_slot = comp["modded_slots"][0]

system_cols = [
    "profit", "profit_margin", "input_cost", "output_value",
    "worker_food_cost", "is_modifier_output", "output_price", "modifier_food_output",
    "epe", "produced", "output", "category",
]

rows_first = []  # PM | Good | Amount | Cost | Pct_of_food

for pm_name, pm in recipe_slot.items():
    victuals = pm.get("output", 0.0) if pm.get("produced") == "victuals" else 0.0
    victuals = float(victuals) if victuals else 0.0
    food_out = victuals * VICTUALS_TO_FOOD

    good_costs = []
    for key, val in pm.items():
        if key in system_cols or key.startswith(("profit", "input_cost", "output_value", "worker_food_cost", "output_price", "is_modifier_output", "epe")):
            continue
        good = key
        amount = float(val) if isinstance(val, (int, float)) and not pd.isna(val) else 0.0
        if amount <= 0:
            continue
        price = 0.0
        if good in goods_data.modded_df.index:
            price = float(goods_data.modded_df.loc[good, "default_market_price"]) if "default_market_price" in goods_data.modded_df.columns else 0.0
        elif good in goods_data.vanilla_df.index:
            price = float(goods_data.vanilla_df.loc[good, "default_market_price"])
        cost = amount * price
        good_costs.append((good, amount, cost))

    total_cost = sum(c for _, _, c in good_costs)
    for good, amount, cost in good_costs:
        pct_of_food = (cost / total_cost * 100) if total_cost > 0 else 0.0
        food_contributed = (pct_of_food / 100) * food_out
        food_per_unit = food_contributed / amount if amount > 0 else np.nan
        food_per_gold = food_contributed / cost if cost > 0 else np.nan
        rows_first.append({
            "PM": pm_name,
            "Input_Good": good,
            "Amount": amount,
            "Cost": cost,
            "Pct_of_food": pct_of_food,
            "Food_contributed": food_contributed,
            "Food_per_unit": food_per_unit,
            "Food_per_gold": food_per_gold,
        })

df_first = pd.DataFrame(rows_first)
# First DF: per-recipe, per-good: Amount, Cost, % of finished food (by cost share)
display(df_first.style.format({"Amount": "{:.3f}", "Cost": "{:.3f}", "Pct_of_food": "{:.1f}%", "Food_contributed": "{:.2f}", "Food_per_unit": "{:.2f}", "Food_per_gold": "{:.2f}"}))
# Pivot: recipes x goods with pct (compact view)
df_recipe_pivot = df_first.pivot_table(index="PM", columns="Input_Good", values="Pct_of_food", aggfunc="first").fillna(0)
display(df_recipe_pivot.style.format("{:.1f}%"))

# Second DF: per-good aggregates of food_per_unit and food_per_gold
df_agg = df_first[df_first["Food_per_unit"].notna() & df_first["Food_per_gold"].notna()]
df_second = df_agg.groupby("Input_Good").agg(
    food_per_unit_avg=("Food_per_unit", "mean"),
    food_per_unit_min=("Food_per_unit", "min"),
    food_per_unit_max=("Food_per_unit", "max"),
    food_per_gold_avg=("Food_per_gold", "mean"),
    food_per_gold_min=("Food_per_gold", "min"),
    food_per_gold_max=("Food_per_gold", "max"),
    n_recipes=("Food_per_unit", "count"),
).round(2)
display(df_second)

,PM,Input_Good,Amount,Cost,Pct_of_food,Food_contributed,Food_per_unit,Food_per_gold
0,pp_cookery_rice_legumes,rice,3.300,3.300,46.7%,46.27,14.02,14.02
1,pp_cookery_rice_legumes,legumes,2.810,2.810,39.8%,39.40,14.02,14.02
2,pp_cookery_rice_legumes,pepper,0.190,0.950,13.5%,13.32,70.11,14.02
3,pp_cookery_rice_fish,rice,3.300,3.300,46.7%,46.27,14.02,14.02
4,pp_cookery_rice_fish,fish,2.810,2.810,39.8%,39.40,14.02,14.02
5,pp_cookery_rice_fish,pepper,0.190,0.950,13.5%,13.32,70.11,14.02
6,pp_cookery_wheat_livestock,wheat,2.620,2.620,34.9%,34.58,13.20,13.20
7,pp_cookery_wheat_livestock,livestock,2.620,3.930,52.4%,51.88,19.80,13.20
8,pp_cookery_wheat_livestock,salt,0.190,0.950,12.7%,12.54,66.00,13.20
9,pp_cookery_millet_fish,millet,2.810,2.810,37.5%,37.09,13.20,13.20


Input_Good,fish,fruit,legumes,livestock,maize,millet,pepper,potato,rice,salt,wheat,wild_game,wool
PM,,,,,,,,,,,,,
pp_cookery_labskaus,21.7%,0.0%,0.0%,64.8%,0.0%,0.0%,0.0%,13.6%,0.0%,0.0%,0.0%,0.0%,0.0%
pp_cookery_maize_livestock,0.0%,0.0%,0.0%,58.4%,28.6%,0.0%,0.0%,13.0%,0.0%,0.0%,0.0%,0.0%,0.0%
pp_cookery_millet_fish,49.9%,0.0%,0.0%,0.0%,0.0%,37.5%,0.0%,0.0%,0.0%,12.7%,0.0%,0.0%,0.0%
pp_cookery_mutton_pease,0.0%,0.0%,23.8%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,13.4%,0.0%,0.0%,62.8%
pp_cookery_pemmican,0.0%,5.3%,0.0%,81.1%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,13.6%,0.0%
pp_cookery_rice_fish,39.8%,0.0%,0.0%,0.0%,0.0%,0.0%,13.5%,0.0%,46.7%,0.0%,0.0%,0.0%,0.0%
pp_cookery_rice_legumes,0.0%,0.0%,39.8%,0.0%,0.0%,0.0%,13.5%,0.0%,46.7%,0.0%,0.0%,0.0%,0.0%
pp_cookery_surstromming,62.4%,0.0%,0.0%,0.0%,0.0%,6.3%,0.0%,0.0%,0.0%,31.3%,0.0%,0.0%,0.0%
pp_cookery_wheat_livestock,0.0%,0.0%,0.0%,52.4%,0.0%,0.0%,0.0%,0.0%,0.0%,12.7%,34.9%,0.0%,0.0%


,food_per_unit_avg,food_per_unit_min,food_per_unit_max,food_per_gold_avg,food_per_gold_min,food_per_gold_max,n_recipes
Input_Good,,,,,,,
fish,13.68,13.20,14.30,13.68,13.20,14.30,4
fruit,14.31,14.31,14.31,14.31,14.31,14.31,1
legumes,14.02,14.01,14.02,14.02,14.01,14.02,2
livestock,20.82,19.80,21.46,13.88,13.20,14.31,4
maize,13.72,13.72,13.72,13.72,13.72,13.72,1
millet,13.20,13.20,13.20,13.20,13.20,13.20,2
pepper,70.11,70.11,70.11,14.02,14.02,14.02,2
potato,14.01,13.72,14.30,14.01,13.72,14.30,2
rice,14.02,14.02,14.02,14.02,14.02,14.02,2


In [32]:
# df_second.sort_values("food_per_unit_avg", ascending=False)
df_second.sort_values("food_per_gold_avg", ascending=False)

,food_per_unit_avg,food_per_unit_min,food_per_unit_max,food_per_gold_avg,food_per_gold_min,food_per_gold_max,n_recipes
Input_Good,,,,,,,
fruit,14.31,14.31,14.31,14.31,14.31,14.31,1
wild_game,14.31,14.31,14.31,14.31,14.31,14.31,1
legumes,14.02,14.01,14.02,14.02,14.01,14.02,2
pepper,70.11,70.11,70.11,14.02,14.02,14.02,2
rice,14.02,14.02,14.02,14.02,14.02,14.02,2
wool,17.51,17.51,17.51,14.01,14.01,14.01,1
potato,14.01,13.72,14.30,14.01,13.72,14.30,2
livestock,20.82,19.80,21.46,13.88,13.20,14.31,4
maize,13.72,13.72,13.72,13.72,13.72,13.72,1


## Tavern

In [33]:
tavern_analysis = analyze_building("tavern")
for df in tavern_analysis: display(df)

In [34]:
# Tavern calculator: set parameters and run to get the same-style DF
victuals = 1.0
food_produced = 60.0
employment = 1.0
v_good = goods_data.get_good("victuals") if goods_data else None
victuals_price = float(v_good.get("default_market_price", 3.0)) if v_good is not None else 3.0
production_efficiency = 0.0

# Tavern output is abstract food; value at FOOD_PRICE (defines), not market price
food_good_name = "food"
food_price = float(defines_data.get_define("NMarket", "FOOD_PRICE", 0.05)) if defines_data else 0.05
pop_info = pop_data.get_pop_type("peasants") if pop_data else None
pop_food_consumption = float(pop_info.get("pop_food_consumption", 0.6)) if pop_info is not None else 0.6

worker_food_cost = employment * pop_food_consumption * food_price
victuals_cost = victuals * victuals_price
input_cost = victuals_cost + worker_food_cost
output_value = food_produced * (1 + production_efficiency) * food_price
profit = output_value - input_cost
margin = (output_value / input_cost) - 1 if input_cost > 0 else 0.0
epe = (input_cost / (food_produced * food_price)) - 1 if (food_produced * food_price) > 0 else 0.0

out_col = f"Out_{food_good_name}"
tavern_calc_df = pd.DataFrame([{
    "Version": "Calculator",
    "PM": "tavern_maintenance",
    "Pop_Type": "peasants",
    "Employment (1k)": employment,
    "EPE": epe,
    "Profit": profit,
    "Margin": margin,
    "Input_Cost": input_cost,
    "Output_Val": output_value,
    "Worker_Food_Cost": worker_food_cost,
    "output_price": food_price,
    "In_victuals": victuals,
    out_col: food_produced * (1 + production_efficiency)
}])

meta = ["Version", "PM", "Pop_Type", "Employment (1k)", "EPE"]
econ = ["Profit", "Margin", "Input_Cost", "Output_Val", "Worker_Food_Cost", "output_price"]
goods = ["In_victuals", out_col]
display(tavern_calc_df[meta + econ + goods].fillna(0))


,Version,PM,Pop_Type,Employment (1k),EPE,Profit,Margin,Input_Cost,Output_Val,Worker_Food_Cost,output_price,In_victuals,Out_food
0,Calculator,tavern_maintenance,peasants,1.00,-0.65,5.85,1.86,3.15,9.00,0.15,0.15,1.00,60.00


In [35]:
# I think the tavern should be +- 0 at food baseline price? This also means that all global food multipliers will benefit tavern and subsistence
